# Лабораторная работа: Генерация изображений

**Номер ИСУ:** 334873

**Остаток от деления на 3:** 334873 % 3 = 1

**Выбранная модель:** Генеративно-состязательная сеть (GAN)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, Subset
import torchvision
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from imblearn.over_sampling import SMOTE
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Используемое устройство: {device}")

torch.manual_seed(42)
np.random.seed(42)

## 1. Загрузка и подготовка данных

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
])

train_dataset_full = torchvision.datasets.FashionMNIST(
    root='./data', train=True, download=True, transform=transform
)
test_dataset = torchvision.datasets.FashionMNIST(
    root='./data', train=False, download=True, transform=transform
)

train_subset_size = 10000
train_indices = np.random.choice(len(train_dataset_full), train_subset_size, replace=False)
train_dataset = Subset(train_dataset_full, train_indices)

class_names = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

batch_size = 256
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"Размер тренировочного набора: {len(train_dataset)}")
print(f"Размер тестового набора: {len(test_dataset)}")
print(f"Размер изображения: 28x28")
print(f"Количество классов: {len(class_names)}")

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    for img, label in train_dataset_full:
        if label == i:
            ax.imshow(img.squeeze(), cmap='gray')
            ax.set_title(class_names[i])
            ax.axis('off')
            break
plt.suptitle('Примеры изображений из Fashion-MNIST', fontsize=14)
plt.tight_layout()
plt.show()

## 2. Автокодировщик

In [ ]:
class Encoder(nn.Module):
    def __init__(self, latent_dim=32):
        super(Encoder, self).__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 32, 3, stride=2, padding=1),
            nn.BatchNorm2d(32),
            nn.LeakyReLU(0.2),
            nn.Conv2d(32, 64, 3, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.LeakyReLU(0.2),
            nn.Conv2d(64, 128, 3, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2),
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, latent_dim)
        )
    
    def forward(self, x):
        return self.encoder(x)


class Decoder(nn.Module):
    def __init__(self, latent_dim=32):
        super(Decoder, self).__init__()
        self.fc = nn.Linear(latent_dim, 128 * 4 * 4)
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(128, 64, 3, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.LeakyReLU(0.2),
            nn.ConvTranspose2d(64, 32, 3, stride=2, padding=1, output_padding=1),
            nn.BatchNorm2d(32),
            nn.LeakyReLU(0.2),
            nn.ConvTranspose2d(32, 1, 3, stride=2, padding=1, output_padding=1),
            nn.Sigmoid()
        )
    
    def forward(self, z):
        x = self.fc(z)
        x = x.view(-1, 128, 4, 4)
        return self.decoder(x)


class Autoencoder(nn.Module):
    def __init__(self, latent_dim=32):
        super(Autoencoder, self).__init__()
        self.encoder = Encoder(latent_dim)
        self.decoder = Decoder(latent_dim)
    
    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z)
    
    def encode(self, x):
        return self.encoder(x)
    
    def decode(self, z):
        return self.decoder(z)


latent_dim = 32
autoencoder = Autoencoder(latent_dim).to(device)
print(autoencoder)

### Обучение автокодировщика

In [ ]:
criterion = nn.MSELoss()
optimizer = optim.Adam(autoencoder.parameters(), lr=1e-3)

num_epochs = 10
train_losses = []
test_losses = []

for epoch in range(num_epochs):
    autoencoder.train()
    epoch_loss = 0
    for images, _ in train_loader:
        images = images.to(device)
        
        optimizer.zero_grad()
        outputs = autoencoder(images)
        loss = criterion(outputs, images)
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
    
    train_loss = epoch_loss / len(train_loader)
    train_losses.append(train_loss)
    
    autoencoder.eval()
    test_loss = 0
    with torch.no_grad():
        for i, (images, _) in enumerate(test_loader):
            if i >= 10:
                break
            images = images.to(device)
            outputs = autoencoder(images)
            loss = criterion(outputs, images)
            test_loss += loss.item()
    
    test_loss = test_loss / min(10, len(test_loader))
    test_losses.append(test_loss)
    
    print(f"Эпоха [{epoch+1}/{num_epochs}], Train Loss: {train_loss:.4f}, Test Loss: {test_loss:.4f}")

### Кривая обучения

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(range(1, num_epochs+1), train_losses, label='Train Loss', marker='o')
plt.plot(range(1, num_epochs+1), test_losses, label='Test Loss', marker='s')
plt.xlabel('Эпоха')
plt.ylabel('MSE Loss')
plt.title('Кривая обучения автокодировщика')
plt.legend()
plt.grid(True)
plt.show()

## 3. Визуализация восстановления изображений

In [ ]:
autoencoder.eval()
with torch.no_grad():
    test_images, test_labels = next(iter(test_loader))
    test_images = test_images[:10].to(device)
    reconstructed = autoencoder(test_images)

fig, axes = plt.subplots(2, 10, figsize=(15, 3))
for i in range(10):
    axes[0, i].imshow(test_images[i].cpu().squeeze(), cmap='gray')
    axes[0, i].axis('off')
    if i == 0:
        axes[0, i].set_title('Оригинал', fontsize=10)
    
    axes[1, i].imshow(reconstructed[i].cpu().squeeze(), cmap='gray')
    axes[1, i].axis('off')
    if i == 0:
        axes[1, i].set_title('Восстановлено', fontsize=10)

plt.suptitle('Восстановление изображений автокодировщиком', fontsize=14)
plt.tight_layout()
plt.show()

## 4. Генерация Гауссовой моделью и SMOTE

### 4.1 Генерация в пространстве изображений

In [ ]:
num_samples = 2000
images_flat = []
labels_list = []

for i, (img, label) in enumerate(train_dataset_full):
    if i >= num_samples:
        break
    images_flat.append(img.numpy().flatten())
    labels_list.append(label)

X_images = np.array(images_flat)
y_labels = np.array(labels_list)

print(f"Размер данных: {X_images.shape}")

### Гауссова модель (PCA)

In [ ]:
pca = PCA(n_components=50)
X_pca = pca.fit_transform(X_images)

gaussian_params = {}
for c in range(10):
    mask = y_labels == c
    X_class = X_pca[mask]
    if len(X_class) > 1:
        mean = np.mean(X_class, axis=0)
        cov = np.cov(X_class.T) + 1e-4 * np.eye(50)
        gaussian_params[c] = {'mean': mean, 'cov': cov}

def generate_gaussian_images(class_id, n_samples=5):
    params = gaussian_params[class_id]
    samples_pca = np.random.multivariate_normal(params['mean'], params['cov'], n_samples)
    samples_images = pca.inverse_transform(samples_pca)
    samples_images = np.clip(samples_images, 0, 1)
    return samples_images.reshape(n_samples, 28, 28)

fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    if i in gaussian_params:
        generated = generate_gaussian_images(i, 1)[0]
        ax.imshow(generated, cmap='gray')
        ax.set_title(class_names[i])
    ax.axis('off')

plt.suptitle('Генерация Гауссовой моделью (пространство изображений, PCA)', fontsize=14)
plt.tight_layout()
plt.show()

### SMOTE в пространстве изображений

In [ ]:
mask = (y_labels == 0) | (y_labels == 1)
X_subset = X_pca[mask][:100]
y_subset = y_labels[mask][:100]

smote = SMOTE(random_state=42, k_neighbors=3)
X_smote, y_smote = smote.fit_resample(X_subset, y_subset)

n_new = len(X_smote) - len(X_subset)
X_new = X_smote[-n_new:] if n_new > 0 else X_smote[-5:]

generated_smote = pca.inverse_transform(X_new[:5])
generated_smote = np.clip(generated_smote, 0, 1).reshape(-1, 28, 28)

fig, axes = plt.subplots(1, 5, figsize=(12, 3))
for i, ax in enumerate(axes):
    ax.imshow(generated_smote[i], cmap='gray')
    ax.axis('off')
plt.suptitle('Генерация SMOTE (пространство изображений, PCA)', fontsize=14)
plt.tight_layout()
plt.show()

### 4.2 Генерация в скрытом пространстве автокодировщика

In [ ]:
autoencoder.eval()
latent_vectors = []
latent_labels = []

with torch.no_grad():
    for images, labels in train_loader:
        images = images.to(device)
        z = autoencoder.encode(images)
        latent_vectors.append(z.cpu().numpy())
        latent_labels.append(labels.numpy())

latent_vectors = np.concatenate(latent_vectors, axis=0)
latent_labels = np.concatenate(latent_labels, axis=0)

print(f"Размер скрытого пространства: {latent_vectors.shape}")

### Гауссова модель в скрытом пространстве

In [ ]:
gaussian_latent_params = {}
for c in range(10):
    mask = latent_labels == c
    z_class = latent_vectors[mask]
    if len(z_class) > 1:
        mean = np.mean(z_class, axis=0)
        cov = np.cov(z_class.T) + 1e-4 * np.eye(latent_dim)
        gaussian_latent_params[c] = {'mean': mean, 'cov': cov}

def generate_gaussian_latent(class_id, n_samples=5):
    params = gaussian_latent_params[class_id]
    z_samples = np.random.multivariate_normal(params['mean'], params['cov'], n_samples)
    z_tensor = torch.FloatTensor(z_samples).to(device)
    with torch.no_grad():
        generated = autoencoder.decode(z_tensor)
    return generated.cpu().numpy().squeeze()

fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    if i in gaussian_latent_params:
        generated = generate_gaussian_latent(i, 1)
        if generated.ndim == 3:
            generated = generated[0]
        ax.imshow(generated, cmap='gray')
        ax.set_title(class_names[i])
    ax.axis('off')

plt.suptitle('Генерация Гауссовой моделью (скрытое пространство)', fontsize=14)
plt.tight_layout()
plt.show()

### SMOTE в скрытом пространстве

In [ ]:
mask = (latent_labels == 0) | (latent_labels == 1)
Z_subset = latent_vectors[mask][:200]
y_subset = latent_labels[mask][:200]

smote_latent = SMOTE(random_state=42, k_neighbors=5)
Z_smote, y_smote = smote_latent.fit_resample(Z_subset, y_subset)

n_new = len(Z_smote) - len(Z_subset)
Z_new = Z_smote[-n_new:] if n_new > 0 else Z_smote[-10:]
y_new = y_smote[-n_new:] if n_new > 0 else y_smote[-10:]

Z_tensor = torch.FloatTensor(Z_new[:10]).to(device)
with torch.no_grad():
    generated_smote_latent = autoencoder.decode(Z_tensor).cpu().numpy().squeeze()

fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    if i < len(generated_smote_latent):
        ax.imshow(generated_smote_latent[i], cmap='gray')
        ax.set_title(f"Class {y_new[i]}")
    ax.axis('off')

plt.suptitle('Генерация SMOTE (скрытое пространство автокодировщика)', fontsize=14)
plt.tight_layout()
plt.show()

## 5. Условный GAN (cGAN)

**Номер ИСУ:** 334873  
**334873 % 3 = 1** → Генеративно-состязательная сеть (GAN)

In [ ]:
class Generator(nn.Module):
    def __init__(self, latent_dim=100, num_classes=10):
        super(Generator, self).__init__()
        self.label_emb = nn.Embedding(num_classes, num_classes)
        
        self.model = nn.Sequential(
            nn.Linear(latent_dim + num_classes, 256),
            nn.LeakyReLU(0.2),
            nn.BatchNorm1d(256),
            nn.Linear(256, 512),
            nn.LeakyReLU(0.2),
            nn.BatchNorm1d(512),
            nn.Linear(512, 1024),
            nn.LeakyReLU(0.2),
            nn.BatchNorm1d(1024),
            nn.Linear(1024, 28 * 28),
            nn.Tanh()
        )
    
    def forward(self, z, labels):
        c = self.label_emb(labels)
        x = torch.cat([z, c], dim=1)
        out = self.model(x)
        return out.view(-1, 1, 28, 28)


class Discriminator(nn.Module):
    def __init__(self, num_classes=10):
        super(Discriminator, self).__init__()
        self.label_emb = nn.Embedding(num_classes, num_classes)
        
        self.model = nn.Sequential(
            nn.Linear(28 * 28 + num_classes, 512),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            nn.Linear(256, 1),
            nn.Sigmoid()
        )
    
    def forward(self, img, labels):
        c = self.label_emb(labels)
        x = img.view(-1, 28 * 28)
        x = torch.cat([x, c], dim=1)
        return self.model(x)


gan_latent_dim = 100
generator = Generator(gan_latent_dim).to(device)
discriminator = Discriminator().to(device)

print("Generator:")
print(generator)
print("\nDiscriminator:")
print(discriminator)

### Обучение cGAN

In [ ]:
criterion = nn.BCELoss()
optimizer_G = optim.Adam(generator.parameters(), lr=0.0002, betas=(0.5, 0.999))
optimizer_D = optim.Adam(discriminator.parameters(), lr=0.0002, betas=(0.5, 0.999))

transform_gan = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])
])

train_dataset_gan_full = torchvision.datasets.FashionMNIST(
    root='./data', train=True, download=True, transform=transform_gan
)

train_dataset_gan = Subset(train_dataset_gan_full, train_indices)
train_loader_gan = DataLoader(train_dataset_gan, batch_size=256, shuffle=True)

num_epochs_gan = 15
g_losses = []
d_losses = []

for epoch in range(num_epochs_gan):
    epoch_g_loss = 0
    epoch_d_loss = 0
    
    for i, (real_images, labels) in enumerate(train_loader_gan):
        batch_size = real_images.size(0)
        real_images = real_images.to(device)
        labels = labels.to(device)
        
        real_labels = torch.ones(batch_size, 1).to(device)
        fake_labels = torch.zeros(batch_size, 1).to(device)
        
        optimizer_D.zero_grad()
        
        outputs = discriminator(real_images, labels)
        d_loss_real = criterion(outputs, real_labels)
        
        z = torch.randn(batch_size, gan_latent_dim).to(device)
        fake_images = generator(z, labels)
        outputs = discriminator(fake_images.detach(), labels)
        d_loss_fake = criterion(outputs, fake_labels)
        
        d_loss = d_loss_real + d_loss_fake
        d_loss.backward()
        optimizer_D.step()
        
        optimizer_G.zero_grad()
        
        outputs = discriminator(fake_images, labels)
        g_loss = criterion(outputs, real_labels)
        
        g_loss.backward()
        optimizer_G.step()
        
        epoch_g_loss += g_loss.item()
        epoch_d_loss += d_loss.item()
    
    avg_g_loss = epoch_g_loss / len(train_loader_gan)
    avg_d_loss = epoch_d_loss / len(train_loader_gan)
    g_losses.append(avg_g_loss)
    d_losses.append(avg_d_loss)
    
    print(f"Эпоха [{epoch+1}/{num_epochs_gan}], G Loss: {avg_g_loss:.4f}, D Loss: {avg_d_loss:.4f}")

### Кривая обучения GAN

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(range(1, num_epochs_gan+1), g_losses, label='Generator Loss', marker='o')
plt.plot(range(1, num_epochs_gan+1), d_losses, label='Discriminator Loss', marker='s')
plt.xlabel('Эпоха')
plt.ylabel('Loss')
plt.title('Кривая обучения cGAN')
plt.legend()
plt.grid(True)
plt.show()

### Визуализация условной генерации

In [ ]:
generator.eval()

fig, axes = plt.subplots(2, 5, figsize=(12, 5))
with torch.no_grad():
    for i, ax in enumerate(axes.flat):
        z = torch.randn(1, gan_latent_dim).to(device)
        label = torch.LongTensor([i]).to(device)
        generated = generator(z, label)
        img = generated.cpu().squeeze().numpy()
        img = (img + 1) / 2
        ax.imshow(img, cmap='gray')
        ax.set_title(class_names[i])
        ax.axis('off')

plt.suptitle('Условная генерация cGAN (по одному изображению каждого класса)', fontsize=14)
plt.tight_layout()
plt.show()

### Генерация нескольких изображений каждого класса

In [ ]:
fig, axes = plt.subplots(10, 6, figsize=(10, 15))
with torch.no_grad():
    for i in range(10):
        for j in range(6):
            z = torch.randn(1, gan_latent_dim).to(device)
            label = torch.LongTensor([i]).to(device)
            generated = generator(z, label)
            img = generated.cpu().squeeze().numpy()
            img = (img + 1) / 2
            axes[i, j].imshow(img, cmap='gray')
            axes[i, j].axis('off')
            if j == 0:
                axes[i, j].set_ylabel(class_names[i], fontsize=8, rotation=0, labelpad=40)

plt.suptitle('Условная генерация cGAN (по 6 изображений каждого класса)', fontsize=14)
plt.tight_layout()
plt.show()

## 6. Генерация изображений из смеси классов

In [ ]:
def generate_mixed_class(generator, class1, class2, alpha=0.5, z=None):
    if z is None:
        z = torch.randn(1, gan_latent_dim).to(device)
    
    emb1 = generator.label_emb(torch.LongTensor([class1]).to(device))
    emb2 = generator.label_emb(torch.LongTensor([class2]).to(device))
    
    mixed_emb = (1 - alpha) * emb1 + alpha * emb2
    
    x = torch.cat([z, mixed_emb], dim=1)
    out = generator.model(x)
    return out.view(1, 1, 28, 28)

class_pairs = [(0, 6), (3, 4), (7, 9), (1, 2)]

fig, axes = plt.subplots(len(class_pairs), 9, figsize=(14, 6))

with torch.no_grad():
    for row, (c1, c2) in enumerate(class_pairs):
        z = torch.randn(1, gan_latent_dim).to(device)
        
        for col, alpha in enumerate(np.linspace(0, 1, 9)):
            generated = generate_mixed_class(generator, c1, c2, alpha, z)
            img = generated.cpu().squeeze().numpy()
            img = (img + 1) / 2
            axes[row, col].imshow(img, cmap='gray')
            axes[row, col].axis('off')
            
            if row == 0:
                axes[row, col].set_title(f'α={alpha:.2f}', fontsize=8)
        
        axes[row, 0].set_ylabel(f'{class_names[c1]}\n↓\n{class_names[c2]}', 
                                fontsize=8, rotation=0, labelpad=50)

plt.suptitle('Интерполяция между классами (смесь классов)', fontsize=14)
plt.tight_layout()
plt.show()

### Генерация из смеси трёх классов

In [ ]:
def generate_mixed_three_classes(generator, classes, weights, z=None):
    if z is None:
        z = torch.randn(1, gan_latent_dim).to(device)
    
    weights = np.array(weights, dtype=np.float32) / sum(weights)
    
    mixed_emb = torch.zeros(1, 10).to(device)
    for cls, w in zip(classes, weights):
        emb = generator.label_emb(torch.LongTensor([cls]).to(device))
        mixed_emb += w * emb
    
    x = torch.cat([z, mixed_emb], dim=1)
    out = generator.model(x)
    return out.view(1, 1, 28, 28)

fig, axes = plt.subplots(2, 5, figsize=(12, 5))
combinations = [
    ([0, 2, 6], [1, 1, 1], 'T-shirt+Pullover+Shirt'),
    ([3, 4, 6], [1, 1, 1], 'Dress+Coat+Shirt'),
    ([7, 9, 5], [1, 1, 1], 'Sneaker+Boot+Sandal'),
    ([1, 3, 4], [1, 1, 1], 'Trouser+Dress+Coat'),
    ([0, 1, 8], [1, 1, 1], 'T-shirt+Trouser+Bag'),
    ([2, 4, 6], [2, 1, 1], 'Pullover(2x)+Coat+Shirt'),
    ([5, 7, 9], [1, 2, 1], 'Sandal+Sneaker(2x)+Boot'),
    ([0, 3, 8], [1, 1, 2], 'T-shirt+Dress+Bag(2x)'),
    ([1, 2, 4], [1, 1, 1], 'Trouser+Pullover+Coat'),
    ([3, 5, 9], [1, 1, 1], 'Dress+Sandal+Boot'),
]

with torch.no_grad():
    for i, (ax, (classes, weights, title)) in enumerate(zip(axes.flat, combinations)):
        generated = generate_mixed_three_classes(generator, classes, weights)
        img = generated.cpu().squeeze().numpy()
        img = (img + 1) / 2
        ax.imshow(img, cmap='gray')
        ax.set_title(title, fontsize=8)
        ax.axis('off')

plt.suptitle('Генерация из смеси трёх классов', fontsize=14)
plt.tight_layout()
plt.show()

## Выводы

1. **Автокодировщик** успешно обучен сжимать изображения Fashion-MNIST в 32-мерное скрытое пространство и восстанавливать их.

2. **Гауссова модель** и **SMOTE** были применены как в пространстве изображений (с использованием PCA), так и в скрытом пространстве автокодировщика. Генерация в скрытом пространстве даёт более качественные результаты.

3. **Условный GAN (cGAN)** был обучен для условной генерации изображений по классам. Модель способна генерировать различные изображения одежды в зависимости от указанного класса.

4. **Генерация из смеси классов** реализована через интерполяцию эмбеддингов классов, что позволяет создавать гибридные изображения.